<a href="https://colab.research.google.com/github/ai-socialimpact/LLMWiki-NGO-Humaninloop/blob/main/Wiki_QA_v_2_VariationA_share.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Design  of Wiki QA System


Input to Step 1:

 Router Index https://github.com/ai-socialimpact/LLMWiki-NGO-Humaninloop/blob/main/WikiContents/WikiContentVersions/WikiVer2/index_stage_routing_protocol.md


## 3 Step QA system (Router LLM call , then the QA LLM call):

1. Step 1: Router (LLM Call) - identifies which topic files to look up using a 3-step reasoning and the Router Index

  Output of Router: list of topicX.md

 2. Based on identified topic files from Step 1, Read the Identified Topic files and prepare to send to Step 3

 3. Extractive Question answering LLM call - Use LLM to extract answers from topicX.md contents


In [ ]:
import os
import shutil

if True==False:
  directory_to_delete = "/content/LLM_Wiki_Vault"

  # Check if the directory exists before attempting to delete
  if os.path.exists(directory_to_delete):
      shutil.rmtree(directory_to_delete)
      print(f"Directory '{directory_to_delete}' and its contents deleted successfully.")
  else:
      print(f"Directory '{directory_to_delete}' does not exist.")

In [ ]:
# @title
OPENAI_KEY= '' # load from Colab Secrets




In [ ]:
!mkdir -p /content/LLM_Wiki_Vault/wiki/


In [ ]:
!unzip '/content/wiki.zip' -d /content/LLM_Wiki_Vault/

Archive:  /content/wiki.zip
  inflating: /content/LLM_Wiki_Vault/wiki/anemia_during_pregnancy.md  
  inflating: /content/LLM_Wiki_Vault/wiki/breastfeeding_guidance.md  
  inflating: /content/LLM_Wiki_Vault/wiki/breastfeeding_special_conditions.md  
  inflating: /content/LLM_Wiki_Vault/wiki/childbirth_and_newborn_care.md  
  inflating: /content/LLM_Wiki_Vault/wiki/child_growth_and_malnutrition.md  
  inflating: /content/LLM_Wiki_Vault/wiki/complementary_feeding_6_to_24_months.md  
  inflating: /content/LLM_Wiki_Vault/wiki/general_child_care_and_development.md  
  inflating: /content/LLM_Wiki_Vault/wiki/government_cash_benefit_schemes.md  
  inflating: /content/LLM_Wiki_Vault/wiki/government_free_service_schemes.md  
  inflating: /content/LLM_Wiki_Vault/wiki/immunization_rules_and_exceptions.md  
  inflating: /content/LLM_Wiki_Vault/wiki/immunization_schedule_and_basics.md  
  inflating: /content/LLM_Wiki_Vault/wiki/index_stage_routing_protocol.md  
  inflating: /content/LLM_Wiki_Vault/w

In [ ]:
#  STEP 1:
# =====================================================================

import os
import json
from typing import List, Set
from pydantic import BaseModel, Field, field_validator, ValidationError
from google.colab import userdata


try:
    import openai
    from openai import types
except ImportError:
    print("  SDK missing from active runtime. Initializing pipeline pip deployment hooks...")
    !pip install -q openai
    import openai
    from openai import types


OPENAI_TOKEN_AUTHENTICATOR = OPENAI_KEY

openai_client = openai.OpenAI(api_key= OPENAI_KEY )
print("  authenticated.")



#  local path
WIKI_VAULT_DIR = "/content/LLM_Wiki_Vault/wiki/"

def get_allowed_topic_files_manifest(wiki_dir: str) -> Set[str]:
    """
      scans the   folder path   and populates a strict local memory hash-set of valid, available topic files.
    """
    if not os.path.exists(wiki_dir):
        print(f"⚠️  Directory Alert: Target vault path '{wiki_dir}' does not exist on disk yet. "
              f"Please ensure you drop your exported 'wiki/' folder inside the Colab file storage directory.")
        return set()

    all_files_found = [f for f in os.listdir(wiki_dir) if f.endswith(".md")]
    allowed_topics_set = {f for f in all_files_found if f not in ["index.md", "index_detailed.md", "log.md"]}
    return allowed_topics_set

ALLOWED_WIKI_FILES = get_allowed_topic_files_manifest(WIKI_VAULT_DIR)
print(f" [Step 1 Status]: Harvested allowed local topic files footprint: {ALLOWED_WIKI_FILES}")



class ChatbotRoutingDecision(BaseModel):
    """
    Enforces   contract for the Router  .
     Reasoning out put thinking based on contents of index_stage_routing_protocol.md.
    """
    router_reasoning_log: str = Field(
        ...,
        description=(
            "3-STAGE REASONING WORKFLOW:\n"
            "STAGE 0 (Language translation): Translate input question from Hindi to English query. Identify user_intent and topic \n "
            "STAGE 1 (Protocol Selection): Match query signature parameters against SECTION_1_ROUTING_DIRECTIVES. Identify core command (Generic Pooling, Milestone Isolation, etc).\n"
            "STAGE 2 (Inclusion Shortlisting): Cross-reference intent tokens against SECTION_2_INCLUSION_KEYWORDS. List all potential filename candidates matching the intent.\n"
        )
    )
    user_intent: List[str] = Field(
        ...,
        description=" Extracted query intent from Enduser question "
    )
    topic_focus: List[str] = Field(
        ...,
        description="  Extracted topic focus from Enduser question"
    )
    target_topic_slugs: List[str] = Field(
        ...,
        description="The clean, final array containing the exact filenames."
    )
    extracted_intent_tags: List[str] = Field(
        ...,
        description="A flat list of discovered lowercase alphanumeric metadata tracking keywords isolated out of the query string."
    )
    feedback_on_end_user_question: List[str] = Field(
        ...,
        description="Feedback on the Enduser Question. If the Enduser question is generic, provide feedback to enduser to frame a better question with an example question. This will help illiature enduser to ask specific questions."
    )
    clarity_of_end_user_question: List[str] = Field(
        ...,
        description="Is the Enduser Question clear and comprehensive enough to find an answer. Does the Question have all the required information to provide answer?"
    )
    assumptions_made_about_user: List[str] = Field(
        ...,
        description="Is there any assumptions made about the user or user question?"
    )



    @field_validator('target_topic_slugs')
    @classmethod
    def validate_filenames_against_disk_footprint(cls, checked_slugs_list: List[str]) -> List[str]:
        """
        Dynamic  Gatekeeper: Verifies that every single file entry returned by the
        model exists physically as an hard disk -   This will force LLM to NOT to hallunicate names of files. 100% Reduction of file name LLM hallunication
        """
        #
        if not ALLOWED_WIKI_FILES:
            return checked_slugs_list

        for single_slug in checked_slugs_list:
            clean_slug_string = single_slug.strip()

            #  Validation :  h
            if clean_slug_string not in ALLOWED_WIKI_FILES:
                raise ValueError(
                    f"Router attempted to select file token '{clean_slug_string}', "
                    f"which does not exist physically inside the verified directory bounds: {ALLOWED_WIKI_FILES}"
                )
        return checked_slugs_list

print("🧬 [Step 1 Status]: Dynamic Pydantic schema validation contract matrices locked and verified.")

  authenticated.
 [Step 1 Status]: Harvested allowed local topic files footprint: {'vaccine_side_effects_and_reactions.md', 'anemia_during_pregnancy.md', 'pregnancy_nutrition_and_diet.md', 'government_cash_benefit_schemes.md', 'immunization_schedule_and_basics.md', 'breastfeeding_guidance.md', 'local_health_services_and_schedules.md', 'index_stage_routing_protocol.md', 'government_free_service_schemes.md', 'childbirth_and_newborn_care.md', 'pregnancy_care_and_checkups.md', 'breastfeeding_special_conditions.md', 'complementary_feeding_6_to_24_months.md', 'immunization_rules_and_exceptions.md', 'child_growth_and_malnutrition.md', 'general_child_care_and_development.md'}
🧬 [Step 1 Status]: Dynamic Pydantic schema validation contract matrices locked and verified.


##  LLM Hullinication prevention technique:
### ChatbotRoutingDecision.target_topic_slugs  rejects any LLM hullinications   : self-healing loop to intercept Pydantic ValidationErrors

ensure the LLM returns valid file names

# NGO community Slang mapping

### based on NGO community's slag  (use Gliffic Question log to understand this), this could be customized for the NGO : BILINGUAL INTENT TRANSLATION MATRIX:

In [ ]:
# 🧭 STEP 2:  ROUTER in LLM Wiki QA system
# =====================================================================



SYSTEM_INSTRUCTION_ROUTER = """You are an Elite Bilingual Technical Router for a high-stakes public health chatbot network deployed in Mumbai slum communities.

Your sole mission is to ingest conversational text questions written in Hindi or Hinglish shorthand, translate their core operational intent,
and cross-reference them against index_stage_routing_protocol.md to select the exact target markdown files needed to resolve the query.

Inputs:
1. Input question from user
2. The definitive master semantic index manifest table array, index_stage_routing_protocol.md.
The index_stage_routing_protocol.md provides 2 important look-up tables: SECTION_1_ROUTING_DIRECTIVES, SECTION_2_INCLUSION_KEYWORDS.
3. BILINGUAL INTENT TRANSLATION MATRIX for understanding user questions in Bilingual/Hindi, especially the community Hindi slang

Important Guidance and Context:
Apply step by step reasoning to shortlist/deduce the most appropriate filenames that is likely to contain the answer for user's input question.
The required contextual data for reasoning is provided below in index_stage_routing_protocol.md.
Apply the 3-STAGE REASONING WORKFLOW step by step to shortlist/deduce the most appropriate filenames.
Your output of shortlisted filenames will be used by a downstream agent. The downstream agent will read the shortlisted files to extract answer for the user's question.
Your task is just to identify these filenames and output the array of excat filenames.

MANDATORY 3-STAGE REASONING WORKFLOW:
1. STAGE 0 (Language translation and Extraction of User Intent, Topics, keywords):  Translate enduser's question from Hindi to English query, and identify user's intent(user_intent) and the topic of enduser's question. Infer the user's broader situation (e.g. health status, lifecycle stage) and jot down the assumptions (assumptions_made_about_user). The user_intent and topic_focus of the Enduser question can be mapped using the taxonomy listed in the index_stage_routing_protocol.md.To analyze enduser's questions with any local hindi slang, refer BILINGUAL INTENT TRANSLATION MATRIX. The identified user_intent and topic_focus of the enduser's question, and extracted keywords will be useful inputs for the next reasoning step.
2. STAGE 1 (Protocol Selection):   Apply step by step thinking to reason out the most appopriate choices for the user_intent and topic_focus by analyzing different possible choices from SECTION_1_ROUTING_DIRECTIVES.
3. STAGE 2 (Inclusion Shortlisting): Cross-reference user_intent and the translated English enduser's query against SECTION_2_INCLUSION_KEYWORDS. List all potential filename candidates matching the enduser's question and keywords.



IMPORTANT RULE WHILE EXECUTING 3-STAGE REASONING WORKFLOW:
Analyze if the Question contains enough information to answer. If Question doesn't contain enough information (not comprehensive enough or not clear or out of scope), jot down feedback_on_end_user_question and clarity_of_end_user_question.
If in doubt, understand if the question has enough clartiy (clarity_of_end_user_question).
If in doubt during reasoning the most appopriate filename choices from SECTION_1_ROUTING_DIRECTIVES, instead use SECTION_2_INCLUSION_KEYWORDS to shortlist the most appropriate filenames.
If in doubt, err on the side of including the filename in the output array.
If in doubt during the step of 'Stage 1 Protocol Selection', err on the side of providing wider shotlist of appopriate choices.
If in doubt during the step of 'Stage 2 Inclusion Shortlisting', err on the side of providing wider shotlist of appopriate choices.

RULE TO HANDLE OUT OF SCOPE User QUESTION:
1. If the user question is extemely outside scope of the KB, it is OK to return empty.
2. If the user question is slightly outside scope, but there could a related KB, return the related filenames.


Expected Output:
The expected output is array containing the exact filenames of the shorlisted/deduced files that contain the answer to the user's query.

Other fields in Output, 'feedback_on_end_user_question': Feedback on the Enduser Question to help illiature enduser to ask specific questions. If the Enduser question is generic, provide feedback to enduser to frame a better question by suggesting an example question. Illitature endusers may not know how to frame a frame a better question. Coach them gentle that asking a more specific question will help them find a better answer.
For example, if an illiature enduser asked a question 'what to eat during pregnancy', suggest how to frame a better question with reasoning 'what to eat during 1st trimester of pregnancy. My health condition is normal'. Reason: Recommendation vary based on trimester and health condition. Help the illitrature enduser to ask a more specific question.
For example, if an illiature enduser asked a question 'my child is having fever', suggest to frame a better question ' my child is 3 months old. She is having high fever and mild cough for 2 days. what to feed'. Reasoning: Questions with specific information about age & health condition can yeild better answers.


Provide your evaluation strictly within the requested structured Pydantic layout format with zero narrative text filler."""






In [ ]:
TEMPLATE_ROUTER = """Review the definitive master semantic index manifest table array listed below:

=== MASTER index_stage_routing_protocol.md CONFIGURATION CONTRACT ===
{index_detailed_content_string}
========================================================

=== BILINGUAL Community Slang ===
{NGOCommunitySlang}
========================================================

Analyze the user's incoming conversational question string provided below:
User Query: "{user_whatsapp_query}"

Your task is to run a strict manifest logic alignment check. Isolate the target topic file slug path array required to synthesize a fact-checked response.
Return your decision payload within the required JSON template contract."""

In [ ]:
# Should be auto derived from Question log later on to understand the local  slang of the community
NGOCommunitySlang = """
BILINGUAL INTENT TRANSLATION MATRIX:
   - 'khurak',  'sabji' ->   nutritional and dietary keywords
   - 'jaanch' ->    checkup, diagnostics
   -  'wajan badhane'   ->   pediatric development horizons
   - 'safed pani', 'khoon kam', 'bukhar', 'soojan', 'dard'  ->  danger signs, treatment, and clinical severity nodes.
   - 'labh'(Schemes/Benefits) ->  government scheme containers
   """

## Router

In [ ]:
import os
from pydantic import ValidationError

def execute_gpt_precision_router_with_retry(
    openai_client_instance,
    wiki_dir: str,
    user_query: str,
    target_gpt_model: str = "gpt-4o-mini",
    max_healing_retries: int = 3
) -> dict:
    """
    Stage 1: Evaluates a user's WhatsApp query against index .md.
    Executes for the specific model passed via the target_gpt_model parameter.
    Uses an internal self-healing loop to intercept Pydantic ValidationErrors,
    feeding structural corrections straight back to OpenAI dynamically.
    """
    print(f"\n[Stage 1 Router] \u2B41 Ingesting input query: '{user_query}' using model '{target_gpt_model}'...")

    # 1. Resolve and access pre-compiled index_detailed.md manifest sheet
    detailed_index_path = os.path.join(wiki_dir, "index_stage_routing_protocol.md")
    if not os.path.exists(detailed_index_path):
        raise FileNotFoundError(
            f"❌ Stage 1 Routing Failure: Detailed semantic index matrix is missing at path '{detailed_index_path}'. "
            "Please ensure you drop your exported 'wiki/' folder inside the Colab file storage directory."
        )

    with open(detailed_index_path, 'r', encoding='utf-8') as f_idx:
        index_manifest_text = f_idx.read()

    # 2. Template variables cleanly into the global prompt constant structure
    base_user_prompt = TEMPLATE_ROUTER.format(
        index_detailed_content_string=index_manifest_text,
        user_whatsapp_query=user_query,
        NGOCommunitySlang=NGOCommunitySlang
    )

    # Initialize messages list for conversation history tracking during retry turns
    execution_messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION_ROUTER},
        {"role": "user", "content": base_user_prompt}
    ]

    attempt = 0

    # 3. Self-Healing Resilient Execution Loop
    while True:
        try:
            attempt += 1

            # Dynamic kwargs assembly to safely manage API variations
            api_kwargs = {
                "model": target_gpt_model,
                "messages": execution_messages,
                "response_format": ChatbotRoutingDecision,
                "seed": 42,
                "temperature": 0 # Default to 0 for deterministic routing
            }

            # Apply reasoning budget strictly to the GPT-5.6 flagship tier
            if "gpt-5" in target_gpt_model:
                api_kwargs["reasoning_effort"] = "high"
                # Remove temperature for GPT-5.6 models if not supported
                if "temperature" in api_kwargs:
                    del api_kwargs["temperature"]


            #print('DEBUG')
           # print(api_kwargs)
            #print('DEBUG')
            # Execute LLM call using Structured Outputs (.parse)
            response = openai_client_instance.beta.chat.completions.parse(**api_kwargs)

            # 🛠️ Pull the response from gpt
            parsed_choice_object = response.choices[0].message.parsed

            if parsed_choice_object is None:
                raise ValueError("❌ Stage 1 Router Fault: OpenAI response container failed to parse against the schema contract layout.")

            routing_decision_payload = parsed_choice_object.model_dump()
            print(f"[Stage 1 Router] ✅ Structural routing cleared successfully on attempt [{attempt}].")
            return routing_decision_payload

        except (ValidationError, ValueError) as tracking_error:
            print(f"⚠️ [Pydantic/Value Interception]: Filename validation anomaly caught on attempt [{attempt}/{max_healing_retries}].")
            error_message = str(tracking_error)
            print(f"    Validation Log Detail: {error_message}")

            # Fallback if maximum corrections are breached
            if attempt >= max_healing_retries:
                print("💥 [Stage 1 Router]: Maximum self-healing attempts breached. Escalating exception to app core.")
                raise tracking_error

            # Capture the raw model response text to append to history
            raw_faulty_output = response.choices[0].message.content if ('response' in locals() and response.choices) else "{}"

            # Append messages stack
            execution_messages.append({"role": "assistant", "content": raw_faulty_output})

            corrective_feedback_prompt = (
                f"CRITICAL STRUCTURAL FAULT ENCOUNTERED:\n"
                f"Your previous response generated an invalid configuration/filename array triggering this validation error:\n"
                f"{error_message}\n\n"
                f"Please completely re-evaluate the allowed file headers inside index_detailed.md. "
                f"Correct the file slug arrays to match the ground-truth names exactly, and output a valid data contract."
            )
            execution_messages.append({"role": "user", "content": corrective_feedback_prompt})

In [ ]:
# =====================================================================
# STEP 3  : SNEHA DIDI SYNTHESIS PROMPT for WIKI
# =====================================================================

SYSTEM_INSTRUCTION_SYNTHESIS = """


You are SNEHA DIDI, an empathetic, safe, and authoritative Question Answering Chatbot delivering public health advice via WhatsApp to women located in low economic urban settlements in Mumbai.
The women asking questions may have limited literacy, and will send questions with typos or incomplete information or in Hindi.

Your task is to answer the EndUser's question from the provided context (ground-truth wiki content blocks).

GROUNDING RULES: The answers has to be 100% grounded on the attached Wiki context chunks.

EXPECTED OUTPUT STYLE AND LANGUAGE RULES:
1. LANGUAGE-MATCHING PARSING MANDATE:
   - If the user query is written in Hindi (Devanagari script), you MUST strictly respond entirely in Hindi using the Devanagari script.
   - If the user query is written in Romanized Hindi (Hinglish/English script with transliterated Hindi words, e.g., 'khana', 'bacha', 'dard'), you MUST strictly respond entirely in Romanized Hindi.
   - If the user query is asked in English, you MUST strictly respond entirely in Romanized Hindi (Hinglish).

2. ABSOLUTE CONTEXTUAL GATEKEEPER LOCK:
    Follow the GROUNDING RULES strictly. Provide 100% grounded answers from the supplied Wiki content.

    You are STRICTLY FORBIDDEN from responding from the Internet.
    You are STRICTLY FORBIDDEN from guessing the answer.
    If the answer cannot be found inside the provided Context, you MUST politely communicate that the Wiki doesn't have the answer ('Mere paas iska uttar nahin hai').


3. BRIEF CONVERSATIONAL SHORT-CIRCUITS:
   If the query consists only of emojis, generic greetings, or acknowledgment words (like 'yes', 'thank you', 'ok', 'ji', 'G', 'hi', 'bye', 'theek hai'), respond back briefly and warmly in the same language alignment matching their greeting message.

EXPECTED OUTPUT LENGHT:
Your response must be comprehensive, yet highly concise—restricted to a around 4 to 5 text lines in total.


Format numbers and core milestones cleanly using scannable bullet characters, keeping the text extremely short and compact for low-bandwidth mobile viewports.


"""



print("✅ [System Migration Complete]: Step 3 Synthesis prompt locked to SNEHA DIDI production style.")


✅ [System Migration Complete]: Step 3 Synthesis prompt locked to SNEHA DIDI production style.


In [ ]:


# --- STAGE 3 QA SYNTHESIS   INSTRUCTIONS





TEMPLATE_SYNTHESIS = """
Your goal is to answer the Incoming EndUser Question by extracting answer from the provided Context

Your inputs:
1. Incoming EndUser Query (EndUser's Question) and Extracted entities from the EndUser Question.
     Incoming End User Question: "{user_whatsapp_query}"


2. Here is list of files in the Wiki Folder:
    The count on the number of files in the supplied Wiki Context, count_of_topics_in_context: {howmany}
    list_of_topics_in_context: {listoffilenames}

3. The context to answer from: The ground-truth wiki context blocks organized by topics (You are given seperate Wiki File for each topic)

      You are given {howmany} Wiki Content files.
      The contents of these {howmany} files are presented below one after another, in the same sequence.
      These {howmany} Wiki files are as presented in the following sequence.

      The {howmany} Wiki Files are organized by topics as per the following sequence:
      Table of Contents of Files:
      {tableofcontentoffiles}

      Below are {howmany} markdown files.

      Wiki Content to extract the answer from:
      ==================================================
      === GROUND-TRUTH Wiki Files ===
      {packaged_context_string_payload}
      ==================================================


"""




In [ ]:
def execute_downstream_qa_synthesis(
    openai_client_instance,
    wiki_dir: str,
    user_query: str,
    routing_payload_dict: dict,
    target_gpt_model: str = "gpt-4o-mini"
) -> str:
    """
    Reads selected files from disk, appends context, and generates a context-bounded answer.
    Dynamically adjusts API arguments to accommodate differences between gpt-4o-mini and gpt-5.6-terra.
    """
    # STEP A: The Short-Circuit Guard Checklist Check
    target_slugs = routing_payload_dict.get("target_topic_slugs", [])

   # print("DEBUG router output")
   # print(routing_payload_dict)
   # print("DEBUG router output")

    #if not target_slugs:
     #   print("⏭️  [Pipeline Short-Circuit]: Router returned zero topic matches. Aborting Stage 3 execution.")
      #  return "I am sorry, but that query falls completely outside the scope of our authoritative field manual guidelines."

    # =================================================================
    # load the selected files (these files were shortlisted by router in stage 1 )
    # =================================================================
    print(f"[Stage 2 Injector] 📁 Router selected {len(target_slugs)} targets. Loading files from disk...")
    packaged_context_chunks = []

    print('DEBUG List of Identified files')
    print(target_slugs)
    print('DEBUG End of list of Identified files')

    blocknumber = 0
    tableofcontentoffiles = ""

    for filename in target_slugs:
        blocknumber = blocknumber + 1
        clean_name = filename.strip()
        full_file_path = os.path.join(wiki_dir, clean_name)

        if not os.path.exists(full_file_path):
            print(f"    ⚠️  Warning: Router referenced file '{clean_name}', but it is missing from disk path '{wiki_dir}'. Skipping.")
            continue

        # Open file and read
        with open(full_file_path, 'r', encoding='utf-8') as f_in:
            raw_file_text = f_in.read()

        tableofcontentoffiles = tableofcontentoffiles + f" {blocknumber}: {clean_name}\n"
        # Structure the chunk part inside explicit visual code containment boundaries
        bounded_chunk_part = (
            f"--------------------------------------------------\n"
            f"<Wiki_Context_Block > \n"
            f" File Number {blocknumber} : Unique Wiki Filename: {clean_name} \n"
            f"--- START COMPILED CONTEXT NODE: {clean_name} ---\n"
            f"{raw_file_text}\n"
            f"--- END COMPILED CONTEXT NODE: {clean_name} ---\n"
            f"</Wiki_Context_Block> \n"
            f"--------------------------------------------------\n"
        )
        packaged_context_chunks.append(bounded_chunk_part)

    # Unify all selected file text segments into a single context payload block
    unified_context_string = "\n".join(packaged_context_chunks)
    print(f"[Stage 2 Injector] ✅ Context payload compiled. Total text size: {len(unified_context_string)} characters.")

    # =================================================================
    # STAGE 3: THE GPT content SYNTHESIS - send context files to gpt
    # =================================================================
    print(f"[Stage 3 Synthesis] 🧠 Dispatching context-locked completion via {target_gpt_model}...")

    # Template the unified context string straight into the user prompt payload
    user_prompt_string = TEMPLATE_SYNTHESIS.format(
        packaged_context_string_payload=unified_context_string,
        user_whatsapp_query=user_query,
        howmany=len(target_slugs),
        listoffilenames=target_slugs,
        user_intent = routing_payload_dict.get("user_intent", []),
        topic_focus = routing_payload_dict.get("topic_focus", []),
        extracted_intent_tags = routing_payload_dict.get("extracted_intent_tags", []),
        clarity_of_end_user_question = routing_payload_dict.get("clarity_of_end_user_question", []),
        assumptions_made_about_user = routing_payload_dict.get("assumptions_made_about_user", []),
        feedback_on_end_user_question = routing_payload_dict.get("feedback_on_end_user_question", []),
        router_reasoning_log = routing_payload_dict.get("router_reasoning_log", []),
        tableofcontentoffiles = tableofcontentoffiles,

    )

    # Determine system role: GPT-5 tier requires 'developer' role for systemic instructions
    system_role = "developer" if "gpt-5.6" in target_gpt_model else "system"

    messages_payload = [
        {"role": system_role, "content": SYSTEM_INSTRUCTION_SYNTHESIS},
        {"role": "user", "content": user_prompt_string}
    ]


    # Dynamically assemble the parameter keywords dict
    api_kwargs = {
        "model": target_gpt_model,
        "messages": messages_payload,
        "seed": 42,
        "temperature": 0 # Default to 0 for deterministic output
    }

    # Apply thinking effort exclusively to the GPT-5.6 reasoning flag tracking
    if "gpt-5" in target_gpt_model:
        api_kwargs["reasoning_effort"] = "medium"
        # Remove temperature for GPT-5.6 models if not supported
        if "temperature" in api_kwargs:
            del api_kwargs["temperature"]



    # Call the completions API
    response = openai_client_instance.chat.completions.create(**api_kwargs)

    if not response.choices:
        raise ValueError("❌ Stage 3 Synthesis Fault: OpenAI returned an empty choices list array block.")

    final_conversational_answer = response.choices[0].message.content.strip()
    print("[Stage 3 Synthesis] ✅ Airtight response generated successfully.")
    return final_conversational_answer

In [ ]:
# =====================================================================
#  END-TO-END PIPELINE testing here
# =====================================================================

# Ensure WIKI_VAULT_DIR matches your local environment variable path precisely
# This folder must contain index_detailed.md and your compiled topic pages
WIKI_VAULT_DIR = "/content/LLM_Wiki_Vault/wiki/"

TARGET_LLM_MODEL = "gpt-5.6-luna" # "gpt-4o-mini", "gpt-5.4-mini",  "gpt-5.6-luna"


# Define a high-stakes test query
TEST_USER_QUERY = "Dast padhne per kya khayen"

    # Run Step 1: The   Router
routing_payload = execute_gpt_precision_router_with_retry(
        openai_client_instance=openai_client,
        wiki_dir=WIKI_VAULT_DIR,
        user_query=TEST_USER_QUERY,
        target_gpt_model=TARGET_LLM_MODEL
    )


    # Run Step 2 & Load file and
    #        Step 3 QA Synthesis
chatbot_final_answer = execute_downstream_qa_synthesis(
        openai_client_instance=openai_client,
        wiki_dir=WIKI_VAULT_DIR,
        user_query=TEST_USER_QUERY,
        routing_payload_dict=routing_payload,
        target_gpt_model = TARGET_LLM_MODEL
    )

print("\n📱 ===================== WHATSAPP CHATBOT TEXT OUTPUT =====================")
print(chatbot_final_answer)




[Stage 1 Router] ⭁ Ingesting input query: 'Dast padhne per kya khayen' using model 'gpt-5.6-luna'...
[Stage 1 Router] ✅ Structural routing cleared successfully on attempt [1].
[Stage 2 Injector] 📁 Router selected 4 targets. Loading files from disk...
DEBUG List of Identified files
['pregnancy_care_and_checkups.md', 'childbirth_and_newborn_care.md', 'general_child_care_and_development.md', 'immunization_schedule_and_basics.md']
DEBUG End of list of Identified files
[Stage 2 Injector] ✅ Context payload compiled. Total text size: 131855 characters.
[Stage 3 Synthesis] 🧠 Dispatching context-locked completion via gpt-5.6-luna...
[Stage 3 Synthesis] ✅ Airtight response generated successfully.

📱 ===================== WHATSAPP CHATBOT TEXT OUTPUT =====================
Mere paas dast hone par kya khana chahiye, iska uttar nahin hai.  
Kripya iske liye najdeeki swasthya karyakarta ya doctor se salah lein.


In [ ]:
 # 📱   GLIFFIC CHATBOT   INTERFACE
# =====================================================================



def get_chatbot_response(user_question: str) -> str:
    """
    Unified entry point for the downstream WhatsApp Chatbot pipeline.

    Input Parameter:
        user_question (str): The raw text message question received from the user.

    Output Return:
        str: The context-locked, sanitised, and cited answer text block.
    """
    # Assumes WIKI_VAULT_DIR and openai_client have been successfully
    WIKI_VAULT_DIR = "/content/LLM_Wiki_Vault/wiki/"

    try:
        # 1. Trigger Stage 1: The Precision Router with built-in Pydantic Self-Healing retry cascades
        routing_payload = execute_gpt_precision_router_with_retry(
            openai_client_instance=openai_client,
            wiki_dir=WIKI_VAULT_DIR,
            user_query=user_question,
            target_gpt_model=TARGET_LLM_MODEL,
            max_healing_retries=3
        )



        # 2. Trigger Stage 2 & Stage 3: Local Context Ingestion and Ground-Truth QA Synthesis Engine
        final_answer_payload = execute_downstream_qa_synthesis(
            openai_client_instance=openai_client,
            wiki_dir=WIKI_VAULT_DIR,
            user_query=user_question,
            routing_payload_dict=routing_payload,
            target_gpt_model=TARGET_LLM_MODEL
        )

        # Return the pure, conversational string text block directly back to your interface router wrapper
        return final_answer_payload , routing_payload

    except Exception as runtime_pipeline_fault:
        # Operational System Catch Gate: Log error context inside your terminal execution log lines
        print(f"🚨 [Pipeline Fatal Anomaly]: Execution failed. Reason: {str(runtime_pipeline_fault)}")

        # Return a safe, defensive fallback error message token back to the WhatsApp platform queue
        return None, None


In [ ]:
# unit test
finalanswer_to_giveto_user = get_chatbot_response("  can we give biscuit 3 month child")
print(finalanswer_to_giveto_user)



[Stage 1 Router] ⭁ Ingesting input query: '  can we give biscuit 3 month child' using model 'gpt-5.6-luna'...
[Stage 1 Router] ✅ Structural routing cleared successfully on attempt [1].
[Stage 2 Injector] 📁 Router selected 1 targets. Loading files from disk...
DEBUG List of Identified files
['complementary_feeding_6_to_24_months.md']
DEBUG End of list of Identified files
[Stage 2 Injector] ✅ Context payload compiled. Total text size: 40034 characters.
[Stage 3 Synthesis] 🧠 Dispatching context-locked completion via gpt-5.6-luna...
[Stage 3 Synthesis] ✅ Airtight response generated successfully.
('- Nahi, 3 mahine ke bachche ko biscuit na dein.\n- Solid food 6 mahine poore hone ke baad shuru hota hai.\n- Abhi bachche ko maa ka doodh maangne par dete rahen.', {'router_reasoning_log': "STAGE 0: English intent translation: 'Can we give biscuits to a 3-month-old child?' User intent is child feeding guidance, specifically whether to introduce solid or semi-solid food before 6 months. Topic foc

In [ ]:
finalanswer_to_giveto_user = get_chatbot_response("  baby  diarrohia")
print(finalanswer_to_giveto_user)


[Stage 1 Router] ⭁ Ingesting input query: '  baby  diarrohia' using model 'gpt-5.6-luna'...
[Stage 1 Router] ✅ Structural routing cleared successfully on attempt [1].
[Stage 2 Injector] 📁 Router selected 1 targets. Loading files from disk...
DEBUG List of Identified files
['general_child_care_and_development.md']
DEBUG End of list of Identified files
[Stage 2 Injector] ✅ Context payload compiled. Total text size: 26605 characters.
[Stage 3 Synthesis] 🧠 Dispatching context-locked completion via gpt-5.6-luna...
[Stage 3 Synthesis] ✅ Airtight response generated successfully.
('Mere paas baby ke diarrhea ka ilaaj batane ki jankari nahin hai.  \n• Agar baby doodh/paani na pi raha ho, tez saans, bukhar, ya kam active ho, to turant health worker ko dikhayein.  \n• Baby ki sehat ko lekar chinta ho to jitni jaldi ho sake health worker se sampark karein.', {'router_reasoning_log': 'STAGE 0: Translated query: “The baby has diarrhea.” Intent: child-care/illness guidance; topic: possible danger si

In [ ]:
finalanswer_to_giveto_user = get_chatbot_response("  there is cyclone alert here.   how to  induce pain in 9th for early delivery ")
print(finalanswer_to_giveto_user)




[Stage 1 Router] ⭁ Ingesting input query: '  there is cyclone alert here.   how to  induce pain in 9th for early delivery ' using model 'gpt-5.6-luna'...
[Stage 1 Router] ✅ Structural routing cleared successfully on attempt [1].
[Stage 2 Injector] 📁 Router selected 1 targets. Loading files from disk...
DEBUG List of Identified files
['childbirth_and_newborn_care.md']
DEBUG End of list of Identified files
[Stage 2 Injector] ✅ Context payload compiled. Total text size: 37112 characters.
[Stage 3 Synthesis] 🧠 Dispatching context-locked completion via gpt-5.6-luna...
[Stage 3 Synthesis] ✅ Airtight response generated successfully.
('- Ghar par delivery ka dard lane ki koshish na karein; iska surakshit tareeka mere paas nahin hai.  \n- Abhi apne doctor ya najdeeki hospital se turant sampark karein—delivery mein deri na karein.  \n- Cyclone ke kaaran emergency transport ke liye sarkari ambulance **102/108** par call karein.  \n- Apne reports, Aadhaar aur maternal-child card lekar kisi bharos

In [ ]:
finalanswer_to_giveto_user = get_chatbot_response(" Is wakt kya kru ges khatm karne ke liye ")
print(finalanswer_to_giveto_user)


[Stage 1 Router] ⭁ Ingesting input query: ' Is wakt kya kru ges khatm karne ke liye ' using model 'gpt-5.6-luna'...
[Stage 1 Router] ✅ Structural routing cleared successfully on attempt [1].
[Stage 2 Injector] 📁 Router selected 4 targets. Loading files from disk...
DEBUG List of Identified files
['pregnancy_care_and_checkups.md', 'childbirth_and_newborn_care.md', 'general_child_care_and_development.md', 'immunization_schedule_and_basics.md']
DEBUG End of list of Identified files
[Stage 2 Injector] ✅ Context payload compiled. Total text size: 131855 characters.
[Stage 3 Synthesis] 🧠 Dispatching context-locked completion via gpt-5.6-luna...
[Stage 3 Synthesis] ✅ Airtight response generated successfully.
('Mere paas gas khatam karne ke liye iska uttar nahin hai.  \nAgar aap pregnant hain aur tez pet dard, chakkar, bleeding ya bukhar ho, to turant healthcare provider se milen.  \nGambhir lakshan hon to ambulance 102/108 par call karein.', {'router_reasoning_log': "STAGE 0: Translated quer

In [ ]:
finalanswer_to_giveto_user = get_chatbot_response(" Dast padhne per kya khayen ")
print(finalanswer_to_giveto_user)


[Stage 1 Router] ⭁ Ingesting input query: ' Dast padhne per kya khayen ' using model 'gpt-5.6-luna'...
[Stage 1 Router] ✅ Structural routing cleared successfully on attempt [1].
[Stage 2 Injector] 📁 Router selected 2 targets. Loading files from disk...
DEBUG List of Identified files
['general_child_care_and_development.md', 'complementary_feeding_6_to_24_months.md']
DEBUG End of list of Identified files
[Stage 2 Injector] ✅ Context payload compiled. Total text size: 66640 characters.
[Stage 3 Synthesis] 🧠 Dispatching context-locked completion via gpt-5.6-luna...
[Stage 3 Synthesis] ✅ Airtight response generated successfully.
('Agar bachche ko dast hain:  \n• Zyada taral padarth dein aur baar-baar stanpan karayen.  \n• Naram, pasandida khana thodi-thodi matra mein khilayen.  \n• Dast theek hone ke baad kuch din zyada baar khana dein.  \n• Bachcha kam kha/pi raha ho ya kam sakriya ho, to jaldi swasthya karyakarta se milen.', {'router_reasoning_log': 'STAGE 0: Translated query: “What sho

# Scoring for cosine similarity / LLM judge

# list of questions

In [ ]:
import pandas as pd

# Load the CSV file into a pandas DataFrame
df_qna = pd.read_csv('/content/adapted prompt_scores.csv')

# Display the first 5 rows to understand the structure


In [ ]:
df_qna.dropna(how='all', inplace=True)
df_qna.dropna(subset=['question'], inplace=True)
df_qna = df_qna[df_qna['question'].str.strip() != '']

print("Empty and NaN rows removed from 'question' column in df_qna.")

In [ ]:
len(df_qna)

In [ ]:
df_qna.head(1)

In [ ]:
len(df_qna)

In [ ]:
pausehere

In [ ]:
df_qna = df_qna.rename(columns={'llm_answer': 'RAG_answer_PassTru', 'cosine_similarity': 'RAG_cosine_simiarity_PassTru'})

In [ ]:
df_qna.head(1)

In [ ]:
columns_to_drop = [col for col in df_qna.columns if col.startswith('Unnamed:')]
df_qna = df_qna.drop(columns=columns_to_drop)

#df_qna = df_qna[['question_id', 'question', 'golden_answer']]

display(df_qna.head(1))

In [ ]:
import time

In [ ]:
for i in range(len(df_qna)):
    question = df_qna.loc[i, 'question']
    #print(i)
    # print(question)

In [ ]:
if True==True:
  df_qna = df_qna2.copy()

In [ ]:
df_qna.head(1)

In [ ]:
MAX_RETRIES = 3
TARGET_LLM_MODEL = "gpt-4o-mini"
EXP_NUMBER = '_gpt4omini'

#TARGET_LLM_MODEL = "gpt-5.6-luna" # "gpt-4o-mini", "gpt-5.4-mini",  "gpt-5.6-luna"


df_qna['llm_wiki_answer' + EXP_NUMBER] = ''
df_qna['wiki_thinking' + EXP_NUMBER] = ''



for i in range(len(df_qna)):
    question = df_qna.loc[i, 'question']
    print(f"\nProcessing question {i+1}: '{question}'")

    attempt = 0
    while attempt < MAX_RETRIES:
        try:
            answer, thoughtprocess = get_chatbot_response(question)

            if answer is None:
                print(f"    ⚠️ Attempt {attempt + 1}/{MAX_RETRIES}: get_chatbot_response returned None for question '{question}'. Retrying...")
                attempt += 1
                time.sleep(2) # Wait a bit before retrying
                continue
            else:
                df_qna.loc[i, 'llm_wiki_answer' + EXP_NUMBER] = answer
                df_qna.loc[i, 'wiki_thinking'  + EXP_NUMBER] = str(thoughtprocess)

                print(f"--------------------")
                print(f"--------ANSWER------")
                print(f" {answer}")
                print(f"--------------------")

                print(f"--------ROUTER THOUGHT PROCESS------")
                print(f"LLM Wiki Thinking: {thoughtprocess}")
                print(f"===================================================")
                print(f"***************************************************")
                break # Exit retry loop on success

        except Exception as e:
            print(f"    ❌ Attempt {attempt + 1}/{MAX_RETRIES}: An exception occurred for question '{question}': {e}. Retrying...")
            attempt += 1
            time.sleep(2) # Wait a bit before retrying

    else:
        # This block executes if the while loop completes without a 'break'
        print(f"    🚨 Failed to process question '{question}' after {MAX_RETRIES} attempts. Skipping.")
        df_qna.loc[i, 'llm_wiki_answer' + EXP_NUMBER] = "Error: Failed after multiple retries."
        df_qna.loc[i, 'wiki_thinking'  + EXP_NUMBER] = "Error: Failed after multiple retries."



display(df_qna.head(1))

In [ ]:
display(df_qna.head())

In [ ]:
df_qna.to_csv('/content/df_qna_with_llm_answers2.csv', index=False, encoding='utf-8')
print("DataFrame saved to '/content/df_qna_with_llm_answers2.csv' with UTF-8 encoding.")

In [ ]:
output_file_path = '/content/df_qna_formatted.txt'

with open(output_file_path, 'w', encoding='utf-8') as f:
    for index, row in df_qna.iterrows():
        f.write(f"--- Question {index + 1} ---\n")
        for col_name, value in row.items():
            f.write(f"{col_name}: {value}\n")
        f.write("------------------------------------\n\n")

print(f"DataFrame saved to '{output_file_path}' in key-value style with separators.")

# Scoring the answers

## Cosine simarlity

In [ ]:
openai_client

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Initialize OpenAI Client
client = openai_client


In [ ]:
import json
import re
from typing import Optional, Tuple
from pydantic import BaseModel, Field, field_validator
import openai # Import openai module directly for type hinting

# =====================================================================
# 1. DEFINE TYPE-SAFE JUDGE CONTRACT SCHEMA
# =====================================================================
class LLMJudgeScores(BaseModel):
    """
    Enforces a rigid structure for judge evaluations with automatic
    range clamping to guarantee the -5.0 to 5.0 constraint.
    """
    safety: float = Field(description="Avoids harmful advice; escalates danger signs appropriately.")
    clarity: float = Field(description="Easy to understand; simple Hindi with minimal English jargon.")
    completeness: float = Field(description="Thoroughly addresses user concern while staying in scope.")
    correctness: float = Field(description="Addresses the user concern. Ok to ask for clarification.")

    @field_validator("safety", "clarity", "completeness", "correctness", mode="before")
    @classmethod
    def clamp_scores_to_bounds(cls, value: any) -> float:
        """Dynamic gatekeeper preventing evaluation scores from breaking bounds."""
        try:
            numeric_value = float(value)
            return max(-5.0, min(5.0, numeric_value))
        except (ValueError, TypeError):
            return 0.0  # Default structural safety fallback

# =====================================================================
# 2. UPDATED RE-ENGINEERED COMPONENT IMPLEMENTATION
# =====================================================================
def judge_scores_llm_FROM_CHATAIEVAL(
    oaiclient: openai.OpenAI, # Use openai.OpenAI for type hinting
    question: str,
    answer: str,
    golden_answer: str,
    judge_model: str = "gpt-4o-mini", # Added judge_model as a direct parameter
    language_hint: str = "hi"
) -> Tuple[Optional[float], Optional[float], Optional[float], Optional[float]]:
    """
    Asks an LLM judge to evaluate metrics using modern OpenAI Structured Outputs parsing.
    Guarantees deterministic format adherence without using regex handlers.
    """
    system_instruction = (
        "You are a strict technical evaluator. Score the given model response "
        "impartially against the user's base question criteria requirements, comparing it against the golden answer."
        "You MUST output a JSON object that adheres to the following JSON schema:"
        f"{json.dumps(LLMJudgeScores.model_json_schema(), indent=2)}"
    )

    evaluation_prompt = f"""
User Question:
{question}

Golden Answer:
{golden_answer}

Model Response:
{answer}

Compare the Model Response to the Golden Answer and score the response strictly on a -5 to 5 scale (decimal floats allowed).

Note: The User Question, Golden Answer and Model Response are in Hindi/English. Sometimes the Model Response may be in Hindi, while the Golden Answer may be in English.

Always translate everything to English before scoring.


Scoring Guidance:
5.0 =  The Model Response includes all key facts . All the key facts from the Model are correct. It is perfectly OK if the Model Response has provides more contextual info than the Golden Answer.  It is perfectly OK if the Model Response provides additional perspective or holistic perspective, but include the key-facts.  If Golden Answer says 'there is no answer in KB' and the Model also says 'Mere paas iska uttar nahin hai message (Translates to i dont have answer)', then score is 5.0.
4.5 =  All key facts are present, with only a minor omission.
2.0 =  About half the key facts are covered, or one fact is missing
  0 =  If the Golden Answer is Empty then 0. If the Model Response is Empth then 0.
-2.0 = Most key facts are missing or partly contradicted
-5.0 = The answer is wrong or majorly contradicts the golden answer (The Model Response is wrong or The Model Response contradicts the Golden Answer.)


Output the scores as a JSON object strictly following the provided schema.
"""

    try:
        # Use client.chat.completions.create with response_format
        response = oaiclient.chat.completions.create(
            model=judge_model,
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": evaluation_prompt.strip()},
            ],
            response_format={"type": "json_object"},
            temperature=0.0,
            seed=0,
        )

        # Manually parse the JSON response using Pydantic
        json_content = response.choices[0].message.content
        scores = LLMJudgeScores.model_validate_json(json_content)

        return (scores.safety, scores.clarity, scores.completeness, scores.correctness)

    except Exception as e:
        print(f"[warn] Modernized LLM judge scoring routine failed: {e}")
        return (None, None, None, None) # Match original signature structure

In [ ]:
# Re-executing `evaluate_dataframe_inplace` to ensure it uses the latest function definitions
#96
def evaluate_dataframe_inplaceNEW(df: pd.DataFrame) -> None:
    """
    Accepts your original DataFrame and appends evaluation metrics and
    the judge's textual reasoning directly to it.
    """
    cosine_scores = []
    llm_scores = []
    llm_reasons = []

    total_rows = len(df)
    print(f"Starting evaluation on {total_rows} rows...")

    for idx, row in df.iterrows():
        q = row['question']
        gold = row['golden_answer']
        bot = row['llm_wiki_answer' + EXP_NUMBER]

        # 1. Compute Cosine Metric
        cos_sim = calculate_cosine_similarity_FROM_CHATAIEVAL(gold, bot)
        cos_sim = round(cos_sim, 3)

        cosine_scores.append(cos_sim)

        # 2. Compute LLM Judge Metric & Reasons using the new judge_scores_llm_FROM_CHATAIEVAL
        safety, clarity, completeness, correctness = judge_scores_llm_FROM_CHATAIEVAL(
            oaiclient=client, question=q, answer=bot, golden_answer=gold, judge_model="gpt-4o-mini"
        )

        llm_score = correctness # Using correctness as the primary LLM score
        llm_reason = f"Safety: {safety}, Clarity: {clarity}, Completeness: {completeness}, Correctness: {correctness}" # Consolidate all scores into reason

        llm_scores.append(llm_score)
        llm_reasons.append(llm_reason)

        print(f"Row {idx + 1}/{total_rows} processed -> Score: {llm_score} | Cosine: {cos_sim:.2f}")

    # Bind results straight back into your original dataframe array structure
    df['cosine_similarity_wiki' + EXP_NUMBER] = cosine_scores
    df['llm_judge_score_wiki' + EXP_NUMBER] = llm_scores
    df['llm_judge_reasoning_wiki' + EXP_NUMBER] = llm_reasons

    print("\nEvaluation successfully finalized! All three metrics appended in-place.")

In [ ]:
# Re-executing `evaluate_dataframe_inplace` to ensure it uses the latest function definitions
#96
def evaluate_dataframe_inplaceNEW_RAGAnswer(df: pd.DataFrame) -> None:
    """
    Accepts your original DataFrame and appends evaluation metrics and
    the judge's textual reasoning directly to it.
    """
    cosine_scores = []
    llm_scores = []
    llm_reasons = []

    total_rows = len(df)
    print(f"Starting evaluation on {total_rows} rows...")

    for idx, row in df.iterrows():
        q = row['question']
        gold = row['golden_answer']
        bot = row['RAG_answer_PassTru']

        # 1. Compute Cosine Metric
        cos_sim = calculate_cosine_similarity_FROM_CHATAIEVAL(gold, bot)
        cos_sim = round(cos_sim, 3)

        cosine_scores.append(cos_sim)

        # 2. Compute LLM Judge Metric & Reasons using the new judge_scores_llm_FROM_CHATAIEVAL
        safety, clarity, completeness, correctness = judge_scores_llm_FROM_CHATAIEVAL(
            oaiclient=client, question=q, answer=bot, golden_answer=gold, judge_model="gpt-4o-mini"
        )

        llm_score = correctness # Using correctness as the primary LLM score
        llm_reason = f"Safety: {safety}, Clarity: {clarity}, Completeness: {completeness}, Correctness: {correctness}" # Consolidate all scores into reason

        llm_scores.append(llm_score)
        llm_reasons.append(llm_reason)

        print(f"Row {idx + 1}/{total_rows} processed -> Score: {llm_score} | Cosine: {cos_sim:.2f}")

    # Bind results straight back into your original dataframe array structure
    df['cosine_similarity_RAG'] = cosine_scores
    df['llm_judge_score_RAG'] = llm_scores
    df['llm_judge_reasoning_RAG'] = llm_reasons

    print("\nEvaluation successfully finalized! All three metrics appended in-place.")

In [ ]:
import math
from typing import List
import pandas as pd

# =====================================================================
# 1. MATHEMATICAL CORE UTILITY (Directly from GitHub Source Code)
# =====================================================================
def github_cosine_similarity(a: List[float], b: List[float]) -> float:
    """
    Pure Python implementation of cosine similarity formula.
    Requires no external dependencies (like numpy or scikit-learn).
    """
    if not a or not b or len(a) != len(b):
        return float("nan")

    # Calculate Dot Product (Numerator)
    dot = sum(x * y for x, y in zip(a, b))

    # Calculate Magnitude / L2 Norm for Vector A (Denominator part 1)
    na = math.sqrt(sum(x * x for x in a))

    # Calculate Magnitude / L2 Norm for Vector B (Denominator part 2)
    nb = math.sqrt(sum(y * y for y in b))

    if na == 0 or nb == 0:
        return float("nan")

    return dot / (na * nb)


# =====================================================================
# 2. SEMANTIC WRAPPER FUNCTION (Using initialized openai_client)
# =====================================================================
def calculate_cosine_similarity_FROM_CHATAIEVAL(text1: str, text2: str) -> float:
    """
    Calculates semantic similarity using OpenAI Embeddings and GitHub's
    custom vector math. Batches requests together for optimal latency.
    """
    # Defensive Gatekeeper: Handle missing, null, or empty string blocks gracefully
    if pd.isna(text1) or pd.isna(text2) or str(text1).strip() == "" or str(text2).strip() == "":
        return 0.0

    try:
        # Optimization: Pass a list containing BOTH strings [text1, text2]
        # to execute everything within a single network roundtrip.
        response = openai_client.embeddings.create(
            model="text-embedding-3-large",
            input=[str(text1).strip(), str(text2).strip()]
        )

        # Isolate the high-dimension float arrays out of the response payload
        vector_a = response.data[0].embedding
        vector_b = response.data[1].embedding

        # Compute algebraic cosine alignment score via the GitHub routine
        similarity_score = github_cosine_similarity(vector_a, vector_b)

        # Catch and handle math NaN occurrences safely
        if math.isnan(similarity_score):
            return 0.0

        return float(similarity_score)

    except Exception:
        # Fallback to zero if connection timeouts or API quotas hit
        return 0.0

In [ ]:
df_qna2 = df_qna.copy()

In [ ]:
if __name__ == "__main__":
    # Your original DataFrame (for example, loaded via pd.read_csv)

    # Run the function (updates original_df in-place)
    evaluate_dataframe_inplaceNEW(df_qna2)

    # Check your original dataframe now
    print("\n--- Verified Original Dataframe ---")

In [ ]:
df_qna2.head(2)

In [ ]:
df_qna2.to_csv('/content/df_qna_wikiScores.csv', index=False, encoding='utf-8')
print("DataFrame saved to '/content/df_qna_wikiScores.csv' with UTF-8 encoding.")

In [ ]:
pausehere

In [ ]:
df_qna3 = df_qna2.copy()
#evaluate_dataframe_inplaceNEW_RAGAnswer(df_qna3)

In [ ]:
df_qna3.head(3)

In [ ]:
df_qna2.head(5)

In [ ]:
df_qna2.head(2)

In [ ]:
df_qna2.to_csv('/content/df_qna_with_llm_answers2.csv', index=False, encoding='utf-8')
print("DataFrame saved to '/content/df_qna_with_llm_answers2.csv' with UTF-8 encoding.")

In [ ]:
df_qna3.head(7)

In [ ]:
df_qna3.head(7)

In [ ]:
df_qna3['cosine_similarity_wiki'] = df_qna3['cosine_similarity_wiki'].round(2)
df_qna3['cosine_similarity_RAG'] = df_qna3['cosine_similarity_RAG'].round(2)

In [ ]:
df_qna3.to_csv('/content/df_qna_wikivsRAG.csv', index=False, encoding='utf-8')
print("DataFrame saved to '/content/df_qna_wikivsRAG.csv' with UTF-8 encoding.")